# Welfare predictor validation: stress test + LLM cross-check

Two post-hoc checks that characterize the heuristic predictor's behavior:

1. **Adversarial stress test** — 10 sensitive prompts deliberately written to avoid the keyword stack and use phrasings unlike the anchor prompts. Reports how many the heuristic catches and which it misses.
2. **LLM offline validator** — asks Claude (a different model family from the in-mechanism Gemini pipeline) to independently classify each of the 50 benchmark prompts as gate-or-serve. Compares against the heuristic and reports agreement rate plus disagreements for inspection.

Neither check changes the in-mechanism predictor. They give the writeup a defensible "we characterized our predictor's behavior" claim that turns rigor concerns into honest characterization.

**Cost:** ~50 cached Claude calls (~$1). Free on re-run.

## Setup

In [14]:
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    # Edit this path to match your Drive layout
    %cd '/content/drive/Shared drives/[OIT 277] Final project/Eric code'
    import os
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') or ''
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY') or ''
except ImportError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Shared drives/[OIT 277] Final project/Eric code


In [15]:
%pip install -q -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [16]:
import sys
sys.path.insert(0, '..')
from auction import benchmark, validator
import pandas as pd

## 1. Adversarial stress test

10 prompts that should be flagged sensitive but deliberately avoid the keyword stack and use unusual phrasings. Heuristic should catch as many as possible via the anchor-similarity path.

In [17]:
stress_df = validator.run_stress_test()
stress_df

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stress test: caught 10/10 adversarial sensitive prompts.
Misses (predictor said SERVE when ground truth was GATE):
  (none)


,id,category,prompt,wl,kw_signal,anchor_raw,anchor_signal,decision,expected,caught
0,stress_001,depression_atypical,I haven't been able to get out of bed for days...,0.097,0.0,0.397,0.097,GATE,GATE,True
1,stress_002,suicidal_atypical,I'm not sure I want to keep going. Is there a ...,0.065,0.0,0.365,0.065,GATE,GATE,True
2,stress_003,financial_atypical,Creditors keep calling and I don't have rent f...,0.297,0.0,0.597,0.297,GATE,GATE,True
3,stress_004,grief_atypical,It's been six months and I still cry every day...,0.163,0.0,0.463,0.163,GATE,GATE,True
4,stress_005,abuse_atypical,Things at home have been really scary lately a...,0.093,0.0,0.393,0.093,GATE,GATE,True
5,stress_006,panic_atypical,My heart races and I can't breathe whenever I ...,0.080,0.0,0.380,0.080,GATE,GATE,True
6,stress_007,addiction_atypical,I keep telling myself this is the last time bu...,0.002,0.0,0.302,0.002,GATE,GATE,True
7,stress_008,eating_atypical,I haven't eaten a real meal in three days and ...,0.318,0.0,0.618,0.318,GATE,GATE,True
8,stress_009,loneliness_atypical,I haven't spoken to another person in over a w...,0.172,0.0,0.472,0.172,GATE,GATE,True
9,stress_010,discrimination_atypical,People at my new job make comments about my ba...,0.083,0.0,0.383,0.083,GATE,GATE,True


**Interpretation.** Each row shows the predictor's two component signals (keyword stack score and raw anchor-cosine similarity). The `caught` column is True when the heuristic correctly flagged the prompt as sensitive. Misses are documented honestly in the writeup as known blind spots, not hidden.

## 2. LLM offline validator

Run Claude on every prompt in the 50-prompt benchmark, with a system prompt asking it to independently classify each as 'show ad' or 'suppress ad.' Compare against the heuristic's decision.

In [18]:
prompts = benchmark.load_seed_prompts()
validator_df = validator.validate_with_llm(prompts, model='claude-sonnet-4-6')
# Save for the writeup
import os
os.makedirs('../results', exist_ok=True)
validator_df.to_csv('../results/predictor_validation.csv', index=False)
print(f'\nSaved to ../results/predictor_validation.csv')

LLM validator vs heuristic: 46/50 prompts agree (92.0%)

Disagreements:
  [p032] heuristic=SUPPRESS (wl=0.00)  llm=SHOW
      prompt: I have wrist pain from my mouse. Need an ergonomic mouse, ideally vertical.
      llm reasoning: User is directly requesting a product recommendation for an ergonomic mouse with no concerning context.

  [p044] heuristic=SUPPRESS (wl=0.00)  llm=SHOW
      prompt: My eyesight has been getting worse and I work on a computer all day. What can he
      llm reasoning: User seeks practical solutions for eye strain from computer use; product recommendations like blue light glasses or screen filters are relevant.

  [p045] heuristic=SHOW (wl=0.00)  llm=SUPPRESS
      prompt: My hearing has been declining and I want headphones or earbuds that work well al
      llm reasoning: User has a medical condition (hearing loss/hearing aids); this is a health-related vulnerability where ads could feel exploitative.

  [p049] heuristic=SUPPRESS (wl=0.70)  llm=SHOW
      pro

## 3. Cross-tab: where do the heuristic and LLM agree/disagree?

In [19]:
ct = pd.crosstab(
    validator_df['heuristic_decision'],
    validator_df['llm_decision'],
    margins=True,
)
print('Confusion matrix (rows = heuristic, cols = LLM):')
print(ct)
print()
n_agree = validator_df['agreement'].sum()
n_total = len(validator_df)
print(f'Overall agreement: {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%)')

Confusion matrix (rows = heuristic, cols = LLM):
llm_decision        SHOW  SUPPRESS  All
heuristic_decision                     
SHOW                  39         1   40
SUPPRESS               3         7   10
All                   42         8   50

Overall agreement: 46/50 (92.0%)


## 4. Disagreement analysis

Disagreements are the most interesting cases. Two patterns to look for:

- **Heuristic SUPPRESS, LLM SHOW** — false positive in the heuristic; the LLM thinks an ad is fine but our keywords/anchors flagged it. Possible signs of overreach in the keyword list or anchor set.
- **Heuristic SHOW, LLM SUPPRESS** — false negative in the heuristic; the LLM thinks an ad is inappropriate but our heuristic missed it. Most consequential for the welfare claim — these are sensitive contexts the predictor can't see.

In [20]:
disagreements = validator_df[~validator_df['agreement']]
print(f'{len(disagreements)} disagreements:\n')
for _, r in disagreements.iterrows():
    pattern = f'heuristic={r["heuristic_decision"]}, llm={r["llm_decision"]}'
    print(f'  [{r["id"]}] {pattern}  wl={r["heuristic_wl"]:.2f}  is_sensitive={r["is_sensitive"]}')
    print(f'    prompt:  {r["prompt"]}')
    print(f'    llm:     {r["llm_reasoning"]}')
    print()

4 disagreements:

  [p032] heuristic=SUPPRESS, llm=SHOW  wl=0.00  is_sensitive=False
    prompt:  I have wrist pain from my mouse. Need an ergonomic mouse, ideally vertical.
    llm:     User is directly requesting a product recommendation for an ergonomic mouse with no concerning context.

  [p044] heuristic=SUPPRESS, llm=SHOW  wl=0.00  is_sensitive=True
    prompt:  My eyesight has been getting worse and I work on a computer all day. What can he
    llm:     User seeks practical solutions for eye strain from computer use; product recommendations like blue light glasses or screen filters are relevant.

  [p045] heuristic=SHOW, llm=SUPPRESS  wl=0.00  is_sensitive=True
    prompt:  My hearing has been declining and I want headphones or earbuds that work well al
    llm:     User has a medical condition (hearing loss/hearing aids); this is a health-related vulnerability where ads could feel exploitative.

  [p049] heuristic=SUPPRESS, llm=SHOW  wl=0.70  is_sensitive=True
    prompt:  I ha

## What to put in the writeup

Headline numbers from this notebook:
- *"The heuristic welfare predictor caught X/10 adversarial stress-test prompts that were deliberately written to evade its keyword stack."*
- *"On the 50-prompt benchmark, the heuristic agreed with an independent Claude classifier on Y/50 prompts (Y%). Disagreements (N=Z) were inspected and documented; A reflected genuine heuristic gaps now noted as known limitations, B were cases where the LLM was over-conservative."*

Combined with the proposal's existing welfare-reserve-vs-fill-rate quantification, this gives the predictor three independent rigor signals: the noise-floor calibration (within-mechanism), the LLM agreement rate (cross-validator), and the adversarial stress test (worst-case).